# 09 - Inference Benchmark (Efficiency Analysis)

Measures parameters, FLOPs, inference latency (GPU and CPU), and peak GPU
memory for every model in `MODEL_REGISTRY` (Notebook 07) -- the efficiency
side of the comparison table, since the KLA challenge explicitly
benchmarks inference time.

In [1]:
import sys, os, time
sys.path.insert(0, "..")

import numpy as np
import torch
import pandas as pd

from models.restoration_net import BaselineNet, DistributionMixtureRestorationNet

device_gpu = "cuda" if torch.cuda.is_available() else None
device_cpu = "cpu"
print("CUDA available:", torch.cuda.is_available())


CUDA available: True


In [ ]:

try:
    from thop import profile as thop_profile
    HAVE_THOP = True
except ImportError:
    HAVE_THOP = False
    print("thop not available -- run: pip install thop --break-system-packages")
    print("Falling back to a manual conv-layer FLOP counter (less precise, still useful for relative comparison).")


thop not available -- run: pip install thop --break-system-packages
Falling back to a manual conv-layer FLOP counter (less precise, still useful for relative comparison).


In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

def manual_flops_estimate(model, input_shape):
    """Rough Conv2d-only FLOP counter (2*Cin*Cout*K*K*Hout*Wout per conv), ignores norms/activations."""
    total = 0
    hooks = []
    shapes = {}

    def hook(module, inp, out):
        shapes[module] = (inp[0].shape, out.shape)

    for m in model.modules():
        if isinstance(m, torch.nn.Conv2d):
            hooks.append(m.register_forward_hook(hook))

    with torch.no_grad():
        model(torch.randn(*input_shape))

    for h in hooks:
        h.remove()

    for m, (in_shape, out_shape) in shapes.items():
        Cout, Cin_per_group, kh, kw = m.weight.shape
        _, _, Hout, Wout = out_shape
        total += 2 * Cin_per_group * Cout * kh * kw * Hout * Wout

    return total


def measure_latency(model, x, device, n_warmup=5, n_runs=20, use_half=False):
    model = model.to(device).eval()
    x = x.to(device)
    if use_half and device == "cuda":
        model = model.half()
        x = x.half()
    if device == "cuda":
        torch.backends.cudnn.benchmark = True  
    with torch.no_grad():
        for _ in range(n_warmup):
            out = model(x)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_runs):
            out = model(x)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = time.time()
    elapsed_ms = (t1 - t0) / n_runs * 1000
    batch_size = x.shape[0]
    throughput = (batch_size * 1000.0 / elapsed_ms) if elapsed_ms > 0 else float("inf")
    return elapsed_ms, throughput


def measure_gpu_memory(model, x):
    if not torch.cuda.is_available():
        return None
    model = model.to("cuda").eval()
    x = x.to("cuda")
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        model(x)
    return torch.cuda.max_memory_allocated() / (1024**2)  # MB


In [4]:
MODELS_TO_BENCH = {
    "Baseline": BaselineNet(),
    "Full (LDMH+FiLM)": DistributionMixtureRestorationNet(use_film=True),
}

INPUT_SHAPE = (1, 1, 128, 128)  # matches the KLA 128->256 SR setting, batch size 1
dummy_input = torch.randn(*INPUT_SHAPE)

rows = []
for name, model in MODELS_TO_BENCH.items():
    n_params = count_params(model)

    if HAVE_THOP:
        macs, _ = thop_profile(model, inputs=(dummy_input,), verbose=False)
        flops = macs * 2
    else:
        flops = manual_flops_estimate(model, INPUT_SHAPE)

    cpu_ms, cpu_throughput = measure_latency(model, dummy_input, "cpu")
    if torch.cuda.is_available():
        gpu_ms, gpu_throughput = measure_latency(model, dummy_input, "cuda")
        gpu_mem = measure_gpu_memory(model, dummy_input)
    else:
        gpu_ms, gpu_throughput, gpu_mem = None, None, None

    rows.append({
        "model": name,
        "params (M)": n_params / 1e6,
        "GFLOPs": flops / 1e9,
        "CPU latency (ms)": cpu_ms,
        "CPU throughput (img/s)": cpu_throughput,
        "GPU latency (ms)": gpu_ms,
        "GPU throughput (img/s)": gpu_throughput,
        "GPU peak mem (MB)": gpu_mem,
    })

    if gpu_ms is None:
        gpu_str = "None"
    else:
        gpu_str = f"{gpu_ms:.2f}ms ({gpu_throughput:.2f} img/s)"

    print(f"{name}: params={n_params/1e6:.2f}M  GFLOPs={flops/1e9:.2f}  "
          f"CPU={cpu_ms:.2f}ms ({cpu_throughput:.2f} img/s)  GPU={gpu_str}")

Baseline: params=0.09M  GFLOPs=3.90  CPU=58.31ms (17.15 img/s)  GPU=16.85ms (59.33 img/s)
Full (LDMH+FiLM): params=0.12M  GFLOPs=4.84  CPU=65.14ms (15.35 img/s)  GPU=18.22ms (54.89 img/s)


In [5]:
bench_df = pd.DataFrame(rows)
bench_df.to_csv("../results/09_inference_benchmark.csv", index=False)
print(bench_df.to_string(index=False))


           model  params (M)   GFLOPs  CPU latency (ms)  CPU throughput (img/s)  GPU latency (ms)  GPU throughput (img/s)  GPU peak mem (MB)
        Baseline    0.086945 3.897569         58.305609               17.151009         16.854012               59.333054          40.421387
Full (LDMH+FiLM)    0.116138 4.844433         65.137887               15.352048         18.218338               54.889748          47.084961


## Batched throughput + FP16 CUDA optimization comparison

Single-image (batch=1) latency understates real deployment throughput --
all inputs here are a fixed 128x128 size, so batching is straightforward
and is the single biggest lever for GPU throughput. This section measures
throughput at a realistic batch size, and compares FP32 vs FP16 (half
precision), which is the main practical CUDA-level optimization applicable
here (alongside `cudnn.benchmark=True`, already enabled inside
`measure_latency`) -- FP16 typically gives a substantial throughput
increase on Tensor-Core GPUs (V100/A100/H100) at negligible quality cost
for a well-trained restoration model.

In [6]:
BATCH_SIZE_THROUGHPUT = 16
throughput_rows = []

if torch.cuda.is_available():
    for name in MODELS_TO_BENCH:
        batch_input = torch.randn(BATCH_SIZE_THROUGHPUT, 1, 128, 128)

        # Build a fresh model instance for each precision so conversions don't interfere
        if name == "Baseline":
            model_fp32 = BaselineNet()
            model_fp16 = BaselineNet()
        else:
            model_fp32 = DistributionMixtureRestorationNet(use_film=True)
            model_fp16 = DistributionMixtureRestorationNet(use_film=True)
        model_fp16.load_state_dict(model_fp32.state_dict())  # identical weights, fair comparison

        ms_fp32, tput_fp32 = measure_latency(model_fp32, batch_input, "cuda", use_half=False)
        ms_fp16, tput_fp16 = measure_latency(model_fp16, batch_input, "cuda", use_half=True)

        throughput_rows.append({
            "model": name, "batch_size": BATCH_SIZE_THROUGHPUT,
            "FP32 latency (ms/batch)": ms_fp32, "FP32 throughput (img/s)": tput_fp32,
            "FP16 latency (ms/batch)": ms_fp16, "FP16 throughput (img/s)": tput_fp16,
            "FP16 speedup": tput_fp16 / tput_fp32 if tput_fp32 > 0 else float("nan"),
        })
        print(f"{name}: FP32={tput_fp32:.1f} img/s  FP16={tput_fp16:.1f} img/s  "
              f"speedup={tput_fp16/tput_fp32:.2f}x")
else:
    print("No CUDA device available in this environment -- batched FP16 throughput test skipped.")
    print("Run this section on the actual GPU (e.g. the judges' H100) for real numbers.")


Baseline: FP32=116.8 img/s  FP16=200.8 img/s  speedup=1.72x
Full (LDMH+FiLM): FP32=108.1 img/s  FP16=186.1 img/s  speedup=1.72x


In [7]:
import pandas as pd
if throughput_rows:
    throughput_df = pd.DataFrame(throughput_rows)
    throughput_df.to_csv("../results/09_throughput_fp16_comparison.csv", index=False)
    print(throughput_df.to_string(index=False))


           model  batch_size  FP32 latency (ms/batch)  FP32 throughput (img/s)  FP16 latency (ms/batch)  FP16 throughput (img/s)  FP16 speedup
        Baseline          16               136.931705               116.846569                79.664576               200.842091      1.718853
Full (LDMH+FiLM)          16               148.034370               108.083008                85.952866               186.148534      1.722274
